# Fase 2 — Fine-tuning DeBERTa-v3-small (Kaggle GPU) — run 2

Notebook DELGADO: la lógica vive en `src/` del repo (testeada en local con pytest).
Cambios vs run 1: **lr 1e-5** (el run 1 sobreajustó tras la época 1), **eval cada 25%**
de los steps (mejor rescate del checkpoint óptimo) y **ensemble con el baseline** al final.

**Setup requerido:**
- Accelerator: GPU T4 x2 o P100
- Internet: ON (Settings)
- Add-ons > Secrets: `HF_TOKEN` = token de escritura de huggingface.co/settings/tokens

In [ ]:
# Mismas versiones que requirements.txt del repo (reproducibilidad)
!pip -q install "transformers==5.13.0" "datasets==5.0.0" "accelerate==1.14.0" "sentencepiece==0.2.1"
!git clone https://github.com/chamjf234/llm-preference-judge.git repo
import sys; sys.path.insert(0, "repo/src")

# El path del CSV varía entre entornos de Kaggle: detectarlo en vez de asumirlo
from pathlib import Path
CSV = str(next(Path("/kaggle/input").rglob("llm-classification-finetuning/train.csv")))
print("train.csv:", CSV)

In [ ]:
import train
out = train.main(train_csv=CSV, out_dir="/kaggle/working/deberta_out")

In [ ]:
# Ensemble con el baseline (cero GPU): los dos modelos cometen errores distintos
# y el promedio de probabilidades casi siempre mejora el log loss de ambos.
import ensemble
res = ensemble.blend_with_baseline(CSV, out["val_proba_tta"])

In [ ]:
# Subir pesos a HF Hub (NO al repo git: regla del portafolio)
from kaggle_secrets import UserSecretsClient
from transformers import AutoModelForSequenceClassification, AutoTokenizer
token = UserSecretsClient().get_secret("HF_TOKEN")
repo_id = "chamjf234/llm-preference-judge-deberta-v3-small"
AutoModelForSequenceClassification.from_pretrained("/kaggle/working/deberta_out").push_to_hub(repo_id, token=token)
AutoTokenizer.from_pretrained("microsoft/deberta-v3-small").push_to_hub(repo_id, token=token)